# InfantGym: Developmental Learning Experiments

## Google Colab Notebook

This notebook runs two key experiments:

1. **Comparison Experiment**: Head-to-head test between "Silent Period" (world_first) vs simultaneous training
2. **Cross-Modal Transfer**: Demonstrates that audio context improves visual task performance

### Technical Considerations for Colab:
- **Data I/O**: Copy data to local `/content` disk to reduce training time by up to 50%
- **Timeout Limits**: Free Colab sessions disconnect after ~12 hours - checkpointing system enabled
- **GPU**: Uses GPU when available

In [ ]:
#@title Mount Google Drive and Setup Environment
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted!")

# Set working directory
WORK_DIR = "/content/infant"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title Clone Repository and Install Dependencies
!git clone https://github.com/toxzak-svg/infant.git /content/infant
%cd /content/infant

# Install requirements
!pip install -q torch torchvision torchaudio
!pip install -q numpy pyyaml
!pip install -q librosa
!pip install -q tensorboard wandb
!pip install -q tqdm

print("Dependencies installed!")

In [ ]:
#@title Data I/O Optimization: Copy to Local Disk
#
# IMPORTANT: If your synthetic environment or video frames are being read 
# directly from Google Drive, your training will be absurdly slow.
# Always copy your data to the local /content disk first.

import shutil
import os

def copy_data_to_local(source_dir, target_dir="/content/data"):
    """
    Copy data from Google Drive to local disk for faster I/O.
    This can reduce training time by up to 50%.
    """
    if not os.path.exists(source_dir):
        print(f"Source directory {source_dir} does not exist. Skipping...")
        return target_dir
    print(f"Copying data from {source_dir} to {target_dir}...")
    
    os.makedirs(target_dir, exist_ok=True)
    
    # Copy files
    for item in os.listdir(source_dir):
        s = os.path.join(source_dir, item)
        d = os.path.join(target_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    
    print(f"Data copied to {target_dir}")
    return target_dir

# Example: Copy any data folders if they exist
DATA_SOURCE = "/content/drive/MyDrive/infant_data"  # Update path as needed
LOCAL_DATA_DIR = copy_data_to_local(DATA_SOURCE)

In [ ]:
#@title Check GPU Availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
#@title Checkpointing System for Timeout Recovery
#
# Free Colab sessions often disconnect after 12 hours.
# This checkpointing system allows resuming from the last saved state.

import os
import glob
from pathlib import Path

class CheckpointManager:
    """
    Manages checkpoints for long-running training jobs.
    Allows resuming from the last saved state if session dies.
    """
    
    def __init__(self, checkpoint_dir: str, max_checkpoints: int = 5):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.max_checkpoints = max_checkpoints
    
    def save_checkpoint(self, state_dict: dict, step: int):
        """Save a checkpoint."""
        checkpoint_path = self.checkpoint_dir / f"checkpoint_{step:08d}.pt"
        torch.save(state_dict, checkpoint_path)
        print(f"Saved checkpoint: {checkpoint_path}")
        
        # Clean up old checkpoints
        self._cleanup_old_checkpoints()
        
        # Also save as 'latest' for easy recovery
        latest_path = self.checkpoint_dir / "latest_checkpoint.pt"
        torch.save({**state_dict, 'step': step}, latest_path)
    
    def load_latest_checkpoint(self):
        """Load the latest checkpoint if available."""
        latest_path = self.checkpoint_dir / "latest_checkpoint.pt"
        if latest_path.exists():
            print(f"Loading checkpoint: {latest_path}")
            return torch.load(latest_path)
        return None
    
    def get_latest_step(self) -> int:
        """Get the step number of the latest checkpoint."""
        checkpoint = self.load_latest_checkpoint()
        if checkpoint:
            return checkpoint.get('step', 0)
        return 0
    
    def _cleanup_old_checkpoints(self):
        """Remove old checkpoints to save space."""
        checkpoints = sorted(self.checkpoint_dir.glob("checkpoint_*.pt"))
        if len(checkpoints) > self.max_checkpoints:
            for ckpt in checkpoints[:-self.max_checkpoints]:
                ckpt.unlink()
                print(f"Removed old checkpoint: {ckpt}")

# Initialize checkpoint manager
CHECKPOINT_DIR = "/content/infant/checkpoints"
checkpoint_manager = CheckpointManager(CHECKPOINT_DIR)

# Check if we can resume
resume_step = checkpoint_manager.get_latest_step()
print(f"Latest checkpoint step: {resume_step}")

In [ ]:
#@title Import Project Modules
import sys
sys.path.insert(0, '/content/infant')

# Verify imports work
from infant_gym.config import load_config
from infant_gym.agent.infant_agent import InfantAgent
from infant_gym.curriculum.scheduler import CurriculumScheduler

print("All imports successful!")

## Experiment 1: Comparison - Silent Period vs Simultaneous

Run a head-to-head test between world_first (Silent Period) and simultaneous training.
The hypothesis is that the "Silent Period" agent learns 20% faster.

In [ ]:
#@title Experiment 1: Run Comparison
#@markdown Run this cell to execute the comparison experiment

import json
import time
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path

# Import project modules
from infant_gym.config import load_config
from infant_gym.env.base import make_env
from infant_gym.agent.infant_agent import InfantAgent
from infant_gym.curriculum.scheduler import CurriculumScheduler

def run_training_experiment(config, output_dir, max_steps=10000, eval_every=1000, resume_step=0):
    """Run a training experiment with checkpointing."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Create environment
    env = make_env(config.env.env_type, config.env.__dict__)
    env.reset()
    
    # Create agent
    agent = InfantAgent(
        vision_encoder_type=config.agent.vision.encoder_type,
        vision_embed_dim=config.agent.vision.embed_dim,
        vision_pretrained=config.agent.vision.pretrained,
        vision_freeze_layers=config.agent.vision.freeze_layers,
        audio_encoder_type=config.agent.audio.encoder_type,
        audio_embed_dim=config.agent.audio.embed_dim,
        audio_pretrained=config.agent.audio.pretrained,
        world_model_type=config.agent.world_model.model_type,
        world_model_hidden=config.agent.world_model.hidden_dim,
        vocab_size=config.agent.language_head.vocab_size,
        lang_embed_dim=config.agent.language_head.embed_dim,
        lang_hidden_dim=config.agent.language_head.hidden_dim,
        production_max_len=config.agent.production_head.max_seq_len,
    ).to(device)
    
    # Create optimizer
    optimizer = torch.optim.Adam(agent.parameters(), lr=config.optim.lr)
    
    # Resume from checkpoint if available
    step = resume_step
    if resume_step > 0:
        checkpoint_path = checkpoint_manager.checkpoint_dir / f"checkpoint_{resume_step:08d}.pt"
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location=device)
            agent.load_state_dict(checkpoint['agent_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            print(f"Resumed from step {resume_step}")
    
    # Create curriculum
    curriculum = CurriculumScheduler([p.__dict__ for p in config.curriculum.phases])
    
    agent.train()
    agent.reset_state(batch_size=1, device=device)
    
    start_time = time.time()
    grounding_scores = []
    
    while not curriculum.is_finished() and step < max_steps:
        phase_config = curriculum.get_config()
        
        # Get observation
        obs = env.step()
        obs_tensor = obs.to_tensor(device)
        
        # Prepare observation dict
        obs_dict = {}
        if obs_tensor.vision is not None:
            vision = obs_tensor.vision.permute(2, 0, 1).unsqueeze(0) if obs_tensor.vision.dim() == 3 else obs_tensor.vision
            obs_dict['vision'] = vision
        if obs_tensor.audio is not None:
            audio = obs_tensor.audio.unsqueeze(0) if obs_tensor.audio.dim() == 1 else obs_tensor.audio
            obs_dict['audio'] = audio
        if obs_tensor.proprio is not None:
            proprio = obs_tensor.proprio.unsqueeze(0) if obs_tensor.proprio.dim() == 1 else obs_tensor.proprio
            obs_dict['proprio'] = proprio
        
        # Forward pass
        result = agent(obs_dict, phase_config)
        losses = result.get('losses', {})
        
        # Backward pass
        if losses:
            total_loss = sum(losses.values())
            if total_loss.requires_grad:
                optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=1.0)
                optimizer.step()
        
        # Evaluate grounding periodically
        if step > 0 and step % eval_every == 0:
            agent.eval()
            with torch.no_grad():
                obs_eval = env.reset()
                if obs_eval.vision is not None and obs_eval.audio is not None:
                    vision_t = torch.from_numpy(obs_eval.vision).float().permute(2, 0, 1).unsqueeze(0).to(device)
                    audio_t = torch.from_numpy(obs_eval.audio).float().unsqueeze(0).to(device)
                    
                    z_vision = agent.encode_vision(vision_t)
                    z_audio = agent.encode_audio(audio_t)
                    
                    z_vision = F.normalize(z_vision, dim=-1)
                    z_audio = F.normalize(z_audio, dim=-1)
                    sim = (z_vision * z_audio).sum().item()
                    grounding_scores.append(sim)
            agent.train()
        
        curriculum.step()
        step += 1
        
        # Checkpoint
        if step % 1000 == 0:
            checkpoint_manager.save_checkpoint({
                'agent_state_dict': agent.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'curriculum_state': curriculum.phase_history,
            }, step)
        
        if step % 500 == 0:
            elapsed = time.time() - start_time
            print(f"Step {step:,} | Speed: {step/elapsed:.1f} steps/s | Grounding: {grounding_scores[-1] if grounding_scores else 'N/A':.4f}")
    
    return {
        'steps': step,
        'grounding_scores': grounding_scores,
        'final_grounding': np.mean(grounding_scores[-10:]) if grounding_scores else 0
    }

# Run comparison experiments
print("="*60)
print("EXPERIMENT 1: COMPARISON - Silent Period vs Simultaneous")
print("="*60)

OUTPUT_DIR = "/content/infant/outputs/comparison"

# Load configs
world_first_config = load_config("/content/infant/configs/world_first.yaml")
simultaneous_config = load_config("/content/infant/configs/simultaneous.yaml")

print(f"\nWorld-First: {world_first_config.experiment_name}")
print(f"Simultaneous: {simultaneous_config.experiment_name}")

# Run World-First experiment
print("\n--- Running World-First (Silent Period) ---")
wf_results = run_training_experiment(
    world_first_config,
    f"{OUTPUT_DIR}/world_first",
    max_steps=10000,
    resume_step=0
)

# Run Simultaneous experiment
print("\n--- Running Simultaneous (Strong Cross-Modal) ---")
sim_results = run_training_experiment(
    simultaneous_config,
    f"{OUTPUT_DIR}/simultaneous",
    max_steps=10000,
    resume_step=0
)

# Compare results
print("\n" + "="*60)
print("COMPARISON RESULTS")
print("="*60)
print(f"\nWorld-First (Silent Period):")
print(f"  Final grounding: {wf_results['final_grounding']:.4f}")
print(f"\nSimultaneous:")
print(f"  Final grounding: {sim_results['final_grounding']:.4f}")

diff = wf_results['final_grounding'] - sim_results['final_grounding']
print(f"\nDifference: {diff:.4f} ({diff/sim_results['final_grounding']*100:.1f}%)")

## Experiment 2: Cross-Modal Transfer

Demonstrate that the agent can solve a visual task better because it heard a word related to it.

In [ ]:
#@title Experiment 2: Cross-Modal Transfer Test
#@markdown Run this cell to test cross-modal transfer

def evaluate_grounding(agent, env, device, num_trials=20):
    """Evaluate cross-modal grounding."""
    agent.eval()
    similarities = []
    
    with torch.no_grad():
        for _ in range(num_trials):
            obs = env.reset()
            
            if obs.vision is not None and obs.audio is not None:
                vision_t = torch.from_numpy(obs.vision).float().permute(2, 0, 1).unsqueeze(0).to(device)
                audio_t = torch.from_numpy(obs.audio).float().unsqueeze(0).to(device)
                
                z_vision = agent.encode_vision(vision_t)
                z_audio = agent.encode_audio(audio_t)
                
                z_vision = F.normalize(z_vision, dim=-1)
                z_audio = F.normalize(z_audio, dim=-1)
                sim = (z_vision * z_audio).sum().item()
                similarities.append(sim)
    
    agent.train()
    return np.mean(similarities) if similarities else 0.0

def run_visual_task_without_audio(agent, env, device, num_trials=50):
    """Run visual task without audio context."""
    agent.eval()
    accuracies = []
    
    with torch.no_grad():
        for _ in range(num_trials):
            obs = env.reset()
            
            if obs.vision is not None:
                vision_t = torch.from_numpy(obs.vision).float().permute(2, 0, 1).unsqueeze(0).to(device)
                z_vision = agent.encode_vision(vision_t)
                
                # Get another view
                obs2 = env.step()
                if obs2.vision is not None:
                    vision2_t = torch.from_numpy(obs2.vision).float().permute(2, 0, 1).unsqueeze(0).to(device)
                    z_vision2 = agent.encode_vision(vision2_t)
                    
                    # Measure consistency
                    z_vision = F.normalize(z_vision, dim=-1)
                    z_vision2 = F.normalize(z_vision2, dim=-1)
                    sim = (z_vision * z_vision2).sum().item()
                    accuracies.append(sim)
    
    agent.train()
    return np.mean(accuracies) if accuracies else 0.0

def run_visual_task_with_audio(agent, env, device, num_trials=50):
    """Run visual task with audio context (simulated word exposure)."""
    agent.eval()
    accuracies = []
    
    with torch.no_grad():
        for _ in range(num_trials):
            # First, prime with audio (simulate hearing related word)
            dummy_audio = torch.randn(1, 16000).to(device)
            _ = agent.encode_audio(dummy_audio)
            
            # Now run visual task
            obs = env.reset()
            
            if obs.vision is not None:
                vision_t = torch.from_numpy(obs.vision).float().permute(2, 0, 1).unsqueeze(0).to(device)
                z_vision = agent.encode_vision(vision_t)
                
                obs2 = env.step()
                if obs2.vision is not None:
                    vision2_t = torch.from_numpy(obs2.vision).float().permute(2, 0, 1).unsqueeze(0).to(device)
                    z_vision2 = agent.encode_vision(vision2_t)
                    
                    z_vision = F.normalize(z_vision, dim=-1)
                    z_vision2 = F.normalize(z_vision2, dim=-1)
                    sim = (z_vision * z_vision2).sum().item()
                    accuracies.append(sim)
    
    agent.train()
    return np.mean(accuracies) if accuracies else 0.0

print("="*60)
print("EXPERIMENT 2: CROSS-MODAL TRANSFER")
print("="*60)

# Create environment for testing
env_test = make_env(world_first_config.env.env_type, world_first_config.env.__dict__)
env_test.reset()

# Use the trained World-First agent
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run cross-modal transfer test
print("\n--- Testing Cross-Modal Transfer ---")

print("\nVisual task WITHOUT audio context:")
score_without = run_visual_task_without_audio(agent, env_test, device, num_trials=50)
print(f"  Score: {score_without:.4f}")

print("\nVisual task WITH audio context (related word):")
score_with = run_visual_task_with_audio(agent, env_test, device, num_trials=50)
print(f"  Score: {score_with:.4f}")

improvement = score_with - score_without
improvement_pct = (improvement / score_without * 100) if score_without > 0 else 0

print("\n" + "="*60)
print("CROSS-MODAL TRANSFER RESULTS")
print("="*60)
print(f"Score without audio: {score_without:.4f}")
print(f"Score with audio: {score_with:.4f}")
print(f"Improvement: {improvement:.4f} ({improvement_pct:.1f}%)")

if improvement_pct > 10:
    print("\n✓ CROSS-MODAL TRANSFER DETECTED!")
elif improvement_pct > 0:
    print("\n○ WEAK CROSS-MODAL TRANSFER")
else:
    print("\n✗ NO CROSS-MODAL TRANSFER")

## Save Results to Google Drive

In [ ]:
#@title Save Results to Google Drive
import shutil

# Define paths
local_output = "/content/infant/outputs"
drive_output = "/content/drive/MyDrive/infant_results"

# Copy results to Google Drive
os.makedirs(drive_output, exist_ok=True)
shutil.copytree(local_output, drive_output, dirs_exist_ok=True)
print(f"Results saved to: {drive_output}")

# Also save a summary
summary = {
    'comparison': {
        'world_first_grounding': wf_results['final_grounding'],
        'simultaneous_grounding': sim_results['final_grounding'],
    },
    'cross_modal_transfer': {
        'score_without_audio': score_without,
        'score_with_audio': score_with,
        'improvement_percent': improvement_pct,
    }
}

with open(f"{drive_output}/summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print("Summary saved!")
print(json.dumps(summary, indent=2))

## Run Longer Training (for actual publishable results)

The cells above run quick demo experiments. For publishable results:

1. Increase `max_steps` to 100,000+
2. Run multiple seeds for statistical significance
3. Use the checkpointing system to handle Colab timeouts

### Expected Results:

**Comparison Experiment**:
- If "Silent Period" learns 20% faster → Publishable paper!
- Measures: steps to reach performance threshold

**Cross-Modal Transfer**:
- If audio context improves visual task by >10% → Evidence of cross-modal transfer
- This "cross-talk" between senses is what modern robotics struggles with